# MCAR Tests

Assessing whether missing data is **Missing Completely At Random (MCAR)**, the assumption most imputation methods quietly depend on.

Two complementary tests are demonstrated here:

- **Little's chi-square test** evaluates the dataset as a whole and returns a single *p*-value.
- **Pairwise *t*-tests** compare each feature's distribution between the rows where another feature is missing versus present, producing an `m x m` matrix.

Beyond significance, this notebook also shows Cohen's *d* effect sizes and human-readable MCAR / not-MCAR labeling.

## Setup

Import the library and the styling helpers.

In [ ]:
import pandas as pd
import os
import sys

sys.path.append("../")

from src.mcartest.mcar_stats_tests import MCARTest
from src.mcartest._dataframe_utils import style_significant, style_label, style_effect

## Load the data

The Diabetes Missing Data set is a small, publicly available table with missing values across several clinical measurements, which makes it a convenient testbed for MCAR diagnostics.


In [ ]:
df = pd.read_csv(
    "https://raw.githubusercontent.com/YBI-Foundation/Dataset/refs/heads/main/Diabetes%20Missing%20Data.csv"
)

df.head()

### Missingness overview

A quick look at how much is missing per column, which sets expectations for how much the pairwise tests will have to work with. Columns with very little missingness produce sparse splits, and some pairwise tests will not have enough samples to run at all.


In [ ]:
df.isna().sum()

## Little's MCAR test

Little's test evaluates the whole dataset at once. The null hypothesis is that the data is MCAR, so a small *p*-value is evidence **against** MCAR.

Note that this is a single verdict for the entire table, which is useful as a headline but tells you nothing about *which* variables drive the result. The pairwise tests below fill that gap.


In [ ]:
mt = MCARTest(method="little")

little_pvalue = mt.little_mcar_test(df)
print(f"Little's MCAR test p-value: {little_pvalue}")

## Pairwise *t*-tests

Each cell `[h, j]` tests whether feature `h` is MCAR with respect to feature `j`, by comparing the values of `j` between the rows where `h` is missing and where `h` is present.

Non-numeric columns are ignored by the test, but we subset explicitly here so the variable names stay honest about what went in.

Below we generate every view we need from the **same** numeric matrix:

- `pvals_raw`: the raw *p*-value matrix
- `labels_mcar`: `"MCAR"` where the null is not rejected
- `labels_not_mcar`: `"not MCAR"` where the null is rejected
- `labels_both`: both verdicts in one matrix, so blank means only "could not test"
- `effects_raw`: effect-size magnitude bands, reported only where MCAR is rejected


In [ ]:
# numeric-only frame (the guard also lives inside the function; this keeps names explicit)
df_num = df.select_dtypes(include="number")

pvals_raw = mt.mcar_t_tests(df_num)
labels_mcar = mt.mcar_t_tests(df_num, label_mcar=True)
labels_not_mcar = mt.mcar_t_tests(df_num, label_not_mcar=True)
labels_both = mt.mcar_t_tests(df_num, label_both=True)

_, effects_raw = mt.mcar_t_tests(df_num, effect_size=True, effect_if_not_mcar=True)

### Filtering

Some pairwise tests cannot run: if a feature has very little missingness, one side of the split may have too few samples, and the *t*-test returns `NaN`.

The important detail is that the mask is built **once**, off the numeric *p*-value matrix, and then applied to every derived view. The labeled matrices contain empty strings rather than `NaN`, so running `.notna()` on them separately would keep different rows and the tables would no longer line up side by side.


In [ ]:
# build the mask ONCE off the numeric p-value matrix
mask = pvals_raw.notna().any(axis=1)

pvals = pvals_raw[mask].T
mcar = labels_mcar[mask].T
not_mcar = labels_not_mcar[mask].T
both = labels_both[mask].T
effects = effects_raw[mask].T

# all five should share the same shape
print(pvals.shape, mcar.shape, not_mcar.shape, both.shape, effects.shape)

## Raw *p*-values

The unshaded matrix. Cells shaded green are those **above** the 0.05 threshold, meaning the null was not rejected and the pair is consistent with MCAR.


In [ ]:
style_significant(pvals)

## MCAR labels

The same matrix with the *p*-values replaced by a verdict. Only the cells consistent with MCAR are labeled; everything else is blank.

Note the ambiguity here: a blank cell means either "the test rejected MCAR" or "the test could not run at all." That is exactly what the combined view below resolves.


In [ ]:
style_label(mcar, mcar_color="green", mcar_text="white")

## Not-MCAR labels

The complement. Only the cells that **reject** MCAR are labeled, which are the pairs where missingness in one feature is associated with the values of another.


In [ ]:
style_label(not_mcar, not_mcar_color="crimson", not_mcar_text="white")

## Both labels combined

Every testable cell now carries a verdict, so a blank cell means one thing only: the test could not run.

This is generally the most readable single view, since nothing is left implicit.

In [ ]:
style_label(
    both,
    mcar_color="green",
    not_mcar_color="crimson",
    mcar_text="white",
    not_mcar_text="white",
)

## Effect sizes

A *p*-value tells you whether an association exists, but it is heavily influenced by sample size: with enough rows, a trivial difference becomes "significant," while a real association in a sparse missing-group may never reach the threshold.

Cohen's *d* tells you **how strong** the association is. Here the magnitudes are reported only where MCAR was rejected, because under MCAR there is no association to quantify in the first place.

Bands follow the conventional cutoffs: negligible (< 0.2), small (0.2 to 0.5), medium (0.5 to 0.8), large (>= 0.8).


In [ ]:
style_effect(
    effects,
    large="darkred",
    medium="red",
    small="gold",
    text_color="black",
)

## Export

The matrices are ordinary pandas DataFrames, so the styled versions export directly to Excel with the colors preserved.


In [ ]:
data_dir = "../data"
os.makedirs(data_dir, exist_ok=True)

style_significant(pvals).to_excel(os.path.join(data_dir, "mcar_pvalues.xlsx"))
style_label(both, mcar_text="white", not_mcar_text="white").to_excel(
    os.path.join(data_dir, "mcar_labels_both.xlsx")
)
style_effect(effects).to_excel(os.path.join(data_dir, "mcar_effect_sizes.xlsx"))

print(f"Written to {data_dir}/")